In [ ]:
%pip install datasets pandas requests


In [ ]:
import json
import datasets
from datasets import load_dataset




meu_dataset = load_dataset('json', data_files={
    'train':'/home/cecilia/Documentos/PIBIC/Fase1/dados_llms/train_novo.json',
    'validation': '/home/cecilia/Documentos/PIBIC/Fase1/dados_llms/val_novo.json',
    'test': '/home/cecilia/Documentos/PIBIC/Fase1/dados_llms/test_novo.json'
})


In [ ]:
import pandas as pd
frases_treino = pd.DataFrame(meu_dataset['train'])
print(frases_treino.head())


In [5]:
import pandas as pd
df_treino = pd.DataFrame(meu_dataset['test'])
X_train = df_treino['input']
y_train = df_treino['target']

In [ ]:
df_treino.head(5)

In [ ]:
y_train.iloc[148]

In [ ]:
lista_aspectos = []
lista_aux = []
for i in y_train:
    len_quad = len(i.split(' |')) // 4
    if len_quad == 1:
        linha_quad = i.split(' |') 
        lista_aspectos.append(linha_quad[0])
    elif len_quad > 1:
        lista_quad = i.split(' [SEP] ')
        for j in lista_quad:
            linha_quad = j.split(' |')
            lista_aux.append(linha_quad[0])
        str_aspectos = ' | '.join(set(lista_aux))
        lista_aspectos.append(str_aspectos)
        lista_aux = []

print(lista_aspectos)

In [ ]:
lista_aspectos = []
for i in y_train:
    quadruplos = i.split(' [SEP] ')
    aspectos_da_frase = []
    for quad in quadruplos:
        termo = quad.split('|')[0].strip()
        if termo:
            aspectos_da_frase.append(termo)
    
    lista_aspectos.append(" | ".join(sorted(list(set(aspectos_da_frase)))))

In [ ]:
import time
import requests
from collections import Counter



models = "gemma3:27b"

OLLAMA_URL = ''


count_requisicoes = 0



lista_frases = []
for frase in X_train:
    lista_frases.append(frase)

respostas_temporarias = []


parte_fixa = """
Your task is to extract the Aspect terms from a sentence.
- Return ONLY the terms separated by ' | '.
- If there is only one, return only it.
- If it is implicit, return 'implicit'.
- Do not explain anything.

Example:
Input: The food was excellent as well as service.
Aspect: food | service
"""



for i, frase_unica in enumerate(lista_frases):
    prompt = f''' {parte_fixa}
    ###Question###
    Input: {frase_unica}.
    Target: 
    '''

    try:
        if count_requisicoes >15: 
            time.sleep(10)
            count_requisicoes=0
    
        response = requests.post(OLLAMA_URL,json={"model": models,"prompt":prompt,"stream": False, "options": {"temperature": 0}},timeout=180)
    
    
        if response.status_code == 200: 
            count_requisicoes+=1
            dados = response.json()teste_ingles
            etiqueta_gerada = dados.get('response', dados.get('message', {}).get('content', '')).strip()
    
            if not etiqueta_gerada:
                print(f"    Rodada {i+1}: Servidor enviou resposta VAZIA. JSON completo: {dados}")
            else:
                print(f"Rodada {i+1}:")
                print(f"Frase: {frase_unica}") 
                print(f"Aspectos: {etiqueta_gerada}")
                print("-" * 50)
                            
                respostas_temporarias.append(etiqueta_gerada)
    
                
                
                
    
        else:
            print(f"Erro na API (Status: {response.status_code}")
    
    
            
           
    
    except Exception as e:
        print(f"Erro na rodada: {response.status_code} ")
        time.sleep(40)
            


In [ ]:
resultados_pibic_final_gemini = []
dataset_final_gemini = meu_dataset['test']


for i in range(len(dataset_final_gemini)):
  frase_original = dataset_final_gemini[i]['input']
  gabarito_oficial = dataset_final_gemini[i]['target']


  resultados_pibic_final_gemini.append({

        "Frase": frase_original,
        "Gabarito": lista_aspectos[i],
        "IA": respostas_temporarias[i]
    })

for res in resultados_pibic_final_gemini:
    print(f"FRASE: {res['Frase']}")
    print(f"ESPERADO: {res['Gabarito']}")
    print(f"IA: {res['IA']}")
    print("-" * 30)
    

In [ ]:
def calcular_todas_as_metricas_corrigido(gabarito_str, ia_str):
    def limpar(texto):
        return set([a.strip().lower() for a in texto.replace('|', ',').split(',') if a.strip()])

    set_gabarito = limpar(gabarito_str)
    set_ia = limpar(ia_str)

    print(f"DEBUG -> GABARITO: {set_gabarito} | IA: {set_ia}")
    set_gabarito = limpar(gabarito_str)
    set_ia = limpar(ia_str)

    if not set_gabarito:
        return {"Precisão": 0, "Recall": 0, "F1-Score": 0, "Acurácia": 0}

    acertos = set_gabarito.intersection(set_ia)
    n_acertos = len(acertos)
    
    recall = n_acertos / len(set_gabarito)
    precisao = n_acertos / len(set_ia) if len(set_ia) > 0 else 0
    f1 = (2 * precisao * recall) / (precisao + recall) if (precisao + recall) > 0 else 0

    acuracia = 1.0 if set_gabarito == set_ia else 0

    return {
        "Precisão": precisao,
        "Recall": recall,
        "F1-Score": f1,
        "Acurácia": acuracia
    }

In [ ]:
len(set(lista_aspectos))

In [ ]:
metricas_detalhadas = []

for i in range(len(respostas_temporarias)):
    m = calcular_todas_as_metricas_corrigido(lista_aspectos[i], respostas_temporarias[i])
    metricas_detalhadas.append(m)

import pandas as pd
df_final = pd.DataFrame(metricas_detalhadas)

print("\n--- MÉDIAS FINAIS DO EXPERIMENTO ---")
print(df_final.mean())